# Replay checkpoint with existing cameras

这个 notebook 会：
1. 加载训练目录下的 `metadata.yaml` 和 checkpoint
2. 构建环境与 PPO 推理函数
3. rollout 一段轨迹
4. 自动读取模型中已有相机并逐个导出视频

In [1]:
from pathlib import Path
import warnings

import jax
import mujoco
import mediapy as media
from omegaconf import OmegaConf
from orbax import checkpoint as ocp

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics

import tensegrity_playground  # noqa: F401, register envs
from mujoco_playground import registry

warnings.filterwarnings(
    "ignore",
    message="Couldn't find sharding info under RestoreArgs.*",
    category=UserWarning,
)

In [5]:
# ===== 修改这里：训练目录 & checkpoint 名 =====
RUN_DIR = Path('/media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8')
CKPT_NAME = 'ckpt_1310720'

# rollout / 渲染参数
SEED = 1
ROLLOUT_STEPS = 1000
RENDER_EVERY = 1
HEIGHT = 600
WIDTH = 800

In [6]:
cfg_path = RUN_DIR / 'metadata.yaml'
ckpt_path = RUN_DIR / 'checkpoints' / CKPT_NAME

assert cfg_path.exists(), f'metadata not found: {cfg_path}'
assert ckpt_path.exists(), f'checkpoint not found: {ckpt_path}'

config = OmegaConf.load(cfg_path)
env_cfg = registry.get_default_config(config.environment_id)
env = registry.load(config.environment_id, config_overrides=env_cfg)

print('Environment:', config.environment_id)
print('Action size:', env.action_size)
print('Observation size:', env.observation_size)

/home/di/anaconda3/envs/tensaur/lib/python3.11/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "mesh_link_body_01_collision" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(


Environment: PleurobotWalk
Action size: 48
Observation size: {'privileged_state': (184,), 'state': (89,)}


In [7]:
# 恢复 checkpoint（通常是 normalizer, policy, value 三部分）
orbax_checkpointer = ocp.PyTreeCheckpointer()
restored = orbax_checkpointer.restore(ckpt_path, item=None)

if isinstance(restored, (tuple, list)) and len(restored) >= 2:
    normalizer_raw, policy_variables = restored[0], restored[1]
else:
    raise ValueError('Unexpected checkpoint format, expected at least (normalizer, policy).')

normalizer_state = running_statistics.RunningStatisticsState(**normalizer_raw)

ppo_net = ppo_networks.make_ppo_networks(
    env.observation_size,
    env.action_size,
    preprocess_observations_fn=running_statistics.normalize,
    **config.agent.network_factory,
)
make_policy = ppo_networks.make_inference_fn(ppo_net)
policy_fn = make_policy((normalizer_state, policy_variables), deterministic=True)

jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_policy = jax.jit(lambda obs, key: policy_fn(obs, key))

print('Checkpoint loaded:', ckpt_path)

/home/di/anaconda3/envs/tensaur/lib/python3.11/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1269: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Checkpoint loaded: /media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8/checkpoints/ckpt_1310720


In [8]:
# rollout
rng = jax.random.key(SEED)
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)
rollout = [state]

for _ in range(ROLLOUT_STEPS):
    rng, act_rng = jax.random.split(rng)
    ctrl, _ = jit_policy(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)
    if bool(state.done):
        break

print('Rollout frames:', len(rollout))

Rollout frames: 1001


In [9]:
# 自动读取模型已有摄像头
camera_names = [
    mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_CAMERA, i)
    for i in range(env.mj_model.ncam)
]

print('Available cameras:')
for c in camera_names:
    print(' -', c)

Available cameras:
 - camera_Pleurobot3_v41_0
 - camera_Pleurobot3_v41_1
 - camera_Pleurobot3_v41_2
 - camera_Pleurobot3_v41_3


In [ ]:
# 渲染配置 + 导出
fps = int(1.0 / env.dt / RENDER_EVERY)
traj = rollout[::RENDER_EVERY]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False

video_dir = RUN_DIR / 'replay_videos'
video_dir.mkdir(parents=True, exist_ok=True)

for camera in camera_names:
    try:
        frames = env.render(
            traj,
            height=HEIGHT,
            width=WIDTH,
            scene_option=scene_option,
            camera=camera,
        )
        out_path = video_dir / f'{CKPT_NAME}_{camera}.mp4'
        media.write_video(out_path.as_posix(), frames, fps=fps)
        print(f'Saved: {out_path}')
    except Exception as e:
        print(f'[Skip] camera={camera}, reason={e}')

print('Done.')

100%|██████████| 1001/1001 [00:36<00:00, 27.74it/s]


Saved: /media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8/replay_videos/ckpt_1310720_camera_Pleurobot3_v41_0.mp4


100%|██████████| 1001/1001 [00:20<00:00, 47.87it/s]


Saved: /media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8/replay_videos/ckpt_1310720_camera_Pleurobot3_v41_1.mp4


100%|██████████| 1001/1001 [00:17<00:00, 57.75it/s]


Saved: /media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8/replay_videos/ckpt_1310720_camera_Pleurobot3_v41_2.mp4


100%|██████████| 1001/1001 [00:17<00:00, 58.08it/s]


Saved: /media/di/4441-E469/gait_turn/260409002156-pleurobotwalk-ppo-8/replay_videos/ckpt_1310720_camera_Pleurobot3_v41_3.mp4
Done.


: 

In [ ]:
# 可选：直接在 notebook 里预览第一个相机
if camera_names:
    sample_camera = camera_names[0]
    sample_video = (RUN_DIR / 'replay_videos' / f'{CKPT_NAME}_{sample_camera}.mp4').as_posix()
    media.show_video(media.read_video(sample_video), fps=fps)